# Reinforcement Learning : Finetuning HalfCheetah-v5 with SZPO from trajectory preferences

In this assignment, we will fine-tune an RL modle that controls HalfCheetah-v5 model using SZPO from trajectory preferences. You will finetune the model to walk and run forward.

## Problem Description
The HalfCheetah-v5 environment is part of the Mujoco environments in https://gymnasium.farama.org/environments/mujoco/half_cheetah/.

[![/assets/half_cheetah_title.gif](./assets/half_cheetah_title.gif)](./assets/half_cheetah_title.gif)
|                 |                                |
|-----------------|--------------------------------|
|Action Space     |Box(-1.0, 1.0, (6,), float32)   |
|Observation Space|Box(-inf, inf, (17,), float64)  |
|import           |gymnasium.make("HalfCheetah-v5")|


Description
---------------------------------------------------

This environment is based on the work of P. Wawrzyński in [“A Cat-Like Robot Real-Time Learning to Run”](http://staff.elka.pw.edu.pl/~pwawrzyn/pub-s/0812_LSCLRR.pdf). The HalfCheetah is a 2-dimensional robot consisting of 9 body parts and 8 joints connecting them (including two paws). The goal is to apply torque to the joints to make the cheetah run forward (right) as fast as possible, with a positive reward based on the distance moved forward and a negative reward for moving backward. The cheetah’s torso and head are fixed, and torque can only be applied to the other 6 joints over the front and back thighs (which connect to the torso), the shins (which connect to the thighs), and the feet (which connect to the shins).

Action Space
-----------------------------------------------------

![./assets/half_cheetah.png](./assets/half_cheetah.png)

The action space is a `Box(-1, 1, (6,), float32)`. An action represents the torques applied at the hinge joints.
| Num | Action                                  | Control Min | Control Max |  Joint | Type (Unit)  |
| --: | --------------------------------------- | ----------: | ----------: |  ----- | ------------ |
|   0 | Torque applied on the back thigh rotor  |          -1 |           1 |  hinge | torque (N m) |
|   1 | Torque applied on the back shin rotor   |          -1 |           1 |  hinge | torque (N m) |
|   2 | Torque applied on the back foot rotor   |          -1 |           1 |  hinge | torque (N m) |
|   3 | Torque applied on the front thigh rotor |          -1 |           1 |  hinge | torque (N m) |
|   4 | Torque applied on the front shin rotor  |          -1 |           1 |  hinge | torque (N m) |
|   5 | Torque applied on the front foot rotor  |          -1 |           1 |  hinge | torque (N m) |



Observation Space
---------------------------------------------------------------

The observation space consists of the following parts (in order):
*   _qpos (8 elements by default):_ Position values of the robot’s body parts.
*   _qvel (9 elements):_ The velocities of these individual body parts (their derivatives).
By default, the observation space is a `Box(-Inf, Inf, (17,), float64)` where the elements are as follows:

| Num | Observation                               | Min  | Max | Joint | Type (Unit)              |
| --: | ----------------------------------------- | ---- | --- | ----- | ------------------------ |
|   0 | z-coordinate of the front tip             | -Inf | Inf | slide | position (m)             |
|   1 | angle of the front tip                    | -Inf | Inf | hinge | angle (rad)              |
|   2 | angle of the back thigh                   | -Inf | Inf | hinge | angle (rad)              |
|   3 | angle of the back shin                    | -Inf | Inf | hinge | angle (rad)              |
|   4 | angle of the back foot                    | -Inf | Inf | hinge | angle (rad)              |
|   5 | angle of the front thigh                  | -Inf | Inf | hinge | angle (rad)              |
|   6 | angle of the front shin                   | -Inf | Inf | hinge | angle (rad)              |
|   7 | angle of the front foot                   | -Inf | Inf | hinge | angle (rad)              |
|   8 | velocity of the x-coordinate of front tip | -Inf | Inf | slide | velocity (m/s)           |
|   9 | velocity of the z-coordinate of front tip | -Inf | Inf | slide | velocity (m/s)           |
|  10 | angular velocity of the front tip         | -Inf | Inf | hinge | angular velocity (rad/s) |
|  11 | angular velocity of the back thigh        | -Inf | Inf | hinge | angular velocity (rad/s) |
|  12 | angular velocity of the back shin         | -Inf | Inf | hinge | angular velocity (rad/s) |
|  13 | angular velocity of the back foot         | -Inf | Inf | hinge | angular velocity (rad/s) |
|  14 | angular velocity of the front thigh       | -Inf | Inf | hinge | angular velocity (rad/s) |
|  15 | angular velocity of the front shin        | -Inf | Inf | hinge | angular velocity (rad/s) |
|  16 | angular velocity of the front foot        | -Inf | Inf | hinge | angular velocity (rad/s) |

In [ ]:
# Import libraries and environmental parameters
import numpy as np
import torch
from IPython import display
import torch.nn as nn
from collections import deque
import numpy as np
import torch
import random
import matplotlib.pyplot as plt
import json
import gymnasium as gym
from preference import cheetah_trajectory_score

STATE_DIM = 17
ACTION_DIM = 6
EP_LEN = 200

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)
env = gym.make("HalfCheetah-v5", max_episode_steps=EP_LEN)
env.reset(seed=0)
SEED = [33, 81, 41, 31, 83]

In [ ]:
# Preparation of actor network and environment simulation

class ContinuousMLP(nn.Module):
    # Actor network for continuous state and action spaces
    def __init__(self, state_dim: int = 17, action_dim: int = 6, hidden_dim: int = 64):
        """
        defines the actor network architecture
        :param state_dim: the observation space dimension
        :param action_dim: the action space dimension
        :param hidden_dim: the hidden layer dimension
        """
        super().__init__()
        self.fc1 = nn.Linear(in_features=state_dim, out_features=hidden_dim)
        self.fc2 = nn.Linear(in_features=hidden_dim, out_features=hidden_dim)
        self.fc3 = nn.Linear(in_features=hidden_dim, out_features=action_dim)
        self._init_weights()
        self.log_std = nn.Parameter(torch.full((action_dim,), -0.5))

    def _init_weights(self, mean=0.0, std=1.0):
        """
        initialize the weights of the network with a normal distribution
        :param mean: mean of the normal distribution
        :param std: std of the normal distribution
        :return:
        """
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=mean, std=std)
                nn.init.normal_(m.bias, mean=mean, std=std)  # or normal_ if you want biases randomized too

    def forward(self, state):
        """
        forward pass of the network
        :param state: the observation batch of shape (batch_size, state_dim)
        :return: logits representing the mean of action distribution and std of action distribution
        """
        scale1 = float(self.fc1.in_features) ** 0.5
        scale2 = float(self.fc2.in_features) ** 0.5
        scale3 = float(self.fc3.in_features) ** 0.5
        x = torch.tanh(self.fc1(state) / scale1)
        x = torch.tanh(self.fc2(x) / scale2)
        logits = self.fc3(x) / scale3
        std = torch.exp(self.log_std).clamp(1e-3, 10.0)
        return logits, std

def simulate(model, env, deterministic=False, seed=None, actor=None):
    """
    simulate the environment with the actor model (base, dpo, szpo)
    :param model: class inherenting the base model
    :param env: halfcheetah-v5 gym environment
    :param deterministic: whether to use deterministic policy
    :param seed: seed for environment initialization
    :param actor: which actor in the model to use
    :return: reward_ep: episode reward, states: list of states, actions: list of actions, log_probs: list of log probabilities
    """
    actor = model.actor if actor is None else actor
    obs, _ = env.reset(seed=seed)
    max_steps = env.spec.max_episode_steps
    reward_ep = 0
    states = []
    actions = []
    log_probs = []
    for _ in range(max_steps):
        states.append(obs.tolist())
        action, log_prob = model.predict(obs, deterministic=deterministic, actor=actor)
        actions.append(action.tolist())
        log_probs.append(float(log_prob))
        obs, reward, terminated, truncated, info = env.step(action)
        reward_ep += reward
        if truncated:
            break
    return reward_ep, states, actions, log_probs

def evaluate(model, env, deterministic=False, SEED=None):
    """
    evaluate the model on the environment
    :param model: class inherenting the base model
    :param env: halfcheetah-v5 gym environment
    :param deterministic: whether to use deterministic policy
    :param SEED: seed of seeds to initialize the environment, use [33, 81, 41, 31, 83] as default
    :return: value: mean episode reward
    """
    max_steps = env.spec.max_episode_steps
    rewards = []
    for seed in SEED:
        obs, _ = env.reset(seed=seed)
        reward_ep = 0
        for _ in range(max_steps):
            action, log_prob = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = env.step(action)
            reward_ep += reward
            if truncated:
                break
        rewards.append(reward_ep)
    value = np.array(rewards)
    return value

def visualize(model, env):
    obs, info = env.reset(seed=31)
    fig, ax = plt.subplots()
    img = ax.imshow(env.render())
    ax.axis("off")

    reward_ep = 0.0
    for _ in range(EP_LEN):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        reward_ep += reward

        img.set_data(env.render())
        display.clear_output(wait=True)
        display.display(fig)

        if terminated or truncated:
            break

    plt.close(fig)
    return reward_ep

def sample_preference(model, env, SEED=None):
    # sample preference of trajectories by interacting the model with the environment
    # returns:
    #   preferences, a list of preference over trajectories from a preference function
    #   returns, oracle returns of the trajectories
    #   states, states of the trajectories sampled by the current policy
    #   actions, actions of the trajectories sampled by the current policy
    #   log_probs, log probabilities of the actions sampled by the current policy
    #   returns_p, return of the positive perturbed policy
    #   returns_n, return of the negative perturbed policy
    seeds = np.random.randint(0, 100, 5).tolist() if SEED is None else SEED
    preferences = []
    returns, states, actions, log_probs = [], [], [], []
    returns_p, returns_n = [], []
    for ep in range(len(seeds)):
        reward, states, actions, log_probs = simulate(model, env, deterministic=True, seed=seeds[ep], actor=model.actor)
        reward_p, states_p, actions_p, log_probs_p = simulate(model, env, deterministic=True, seed=seeds[ep], actor=model.actor_p)
        reward_n, states_n, actions_n, log_probs_n = simulate(model, env, deterministic=True, seed=seeds[ep], actor=model.actor_n)

        score_p = cheetah_trajectory_score(np.array(states_p), env.unwrapped.dt)
        score_n = cheetah_trajectory_score(np.array(states_n), env.unwrapped.dt)

        preferences.append(np.sign(score_p - score_n))

        returns.append(reward)
        states.extend(states_p)
        actions.extend(actions_p)
        log_probs.extend(log_probs_p)
        returns_p.append(reward_p)
        returns_n.append(reward_n)
    return preferences, returns, states, actions, log_probs, returns_p, returns_n


In class, we talked about stochastic zeroth-order (sign) policy gradient, and the estimator of gradient using a perturbed actor, the steps are as follows:
* perturb the current actor to obtain a perturbed actor
* sample trajectories from the perturbed actor and the current actor
* use human preference to see if the perturbation direction is a improving direction, and estimate the zeroth-order gradient by the preferences between the perturbed and the current actor.
* if yes, update using the estimated gradient.

We will implement an improved version with both positive and negative perturbations. It works as follows:
* perturb the current actor to obtain a positive perturbed actor, and then, perturb to the exact opposite direction to obtain the negative perturbed actor.
* sample trajectories from the both positive and negative perturbed
* use human preference to see which perturbation direction is a policy improving direction and update the gradient.

You will complete both the perturbation part (SZPO.perturb()), and the gradient update part (SZPO.train())

In [ ]:
# Algorithm class
class Base:
    def __init__(self, state_dim: int = 17, action_dim: int = 6, hidden_dim: int = 64, device=None):
        self.state_dim = state_dim          # dimension of the state space
        self.action_dim = action_dim        # dimension of the action space
        self.hidden_dim = hidden_dim        # hidden layer dimension
        self.eps = 1e-6
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") if device is None else device
        self.actor = ContinuousMLP(state_dim, action_dim, hidden_dim).to(self.device)       # actor network

    def save(self, pth):
        # save the actor network parameters to pth
        torch.save(self.actor.state_dict(), pth)

    def load(self, pth):
        # load the actor network parameters from pth
        self.actor.load_state_dict(torch.load(pth, map_location=self.device, weights_only=True))

    def predict(self, state, deterministic=False, actor=None):
        """
        predict the action and log probability of the action given the state using the actor network
        :param state: np.array of shape (state_dim,)
        :param deterministic: whether to use deterministic policy
        :param actor: actor network to use
        :return: action: np.array of shape (action_dim,) and log_prob: float
        """
        actor = self.actor if actor is None else actor
        state = torch.from_numpy(np.atleast_2d(state)).float().to(self.device)
        with torch.no_grad():
            mean, std = actor(state)
            if deterministic:
                dist = torch.distributions.Normal(mean, std)
                u = mean
                a = torch.tanh(u)
            else:
                dist = torch.distributions.Normal(mean, std)
                u = dist.sample()
                a = torch.tanh(u)
            log_prob_u = dist.log_prob(u).sum(dim=-1)
            log_det_tanh = torch.log(1.0 - a.pow(2) + self.eps).sum(dim=-1)
            log_prob = (log_prob_u - log_det_tanh).cpu().numpy().squeeze()
            action = a.cpu().numpy().squeeze()
        return action, log_prob

    def log_probs(self, states, actions, actor=None):
        """
        compute the log probability of the actions given the states using the actor network
        :param states: torch.tensor of shape (batch_size, state_dim)
        :param actions: torch.tensor of shape (batch_size, action_dim)
        :param actor: which actor network to use
        :return: log_probs: torch.tensor of shape (batch_size,)
        """
        actor = self.actor if actor is None else actor
        mean, std = actor(states)
        dist = torch.distributions.Normal(mean, std)

        a = torch.clamp(actions, -1.0 + self.eps, 1.0 - self.eps)
        u = torch.atanh(a)

        log_prob_u = dist.log_prob(u).sum(dim=-1)
        log_det_tanh = torch.log(1.0 - a.pow(2) + self.eps).sum(dim=-1)
        log_probs = log_prob_u - log_det_tanh
        return log_probs

    def freeze_feature_layers(self, actor=None, freeze_std=True):
        """
        freeze the feature layers of the actor network and only train the last layer's weight and bias
        :param actor: which actor network to use
        :param freeze_std: whether to freeze the std of the action distribution
        :return:
        """
        actor = self.actor if actor is None else actor
        for p in actor.parameters():
            p.requires_grad = False
        # Identify the last Linear layer in the actor
        last_linear = None
        for m in actor.modules():
            if isinstance(m, nn.Linear):
                last_linear = m
        trainable_names = []

        # Unfreeze the last layer's weight and bias (if found)
        if last_linear is not None:
            for name, p in actor.named_parameters():
                if (p is last_linear.weight) or (p is last_linear.bias):
                    p.requires_grad = True
                    trainable_names.append(name)

        # keep log_std trainable as part of the output head if not frozen
        if (not freeze_std) and hasattr(actor, 'log_std'):
            log_std_param = getattr(actor, 'log_std')
            if isinstance(log_std_param, torch.nn.Parameter):
                log_std_param.requires_grad = True
                # record its parameter name for return
                for name, p in actor.named_parameters():
                    if p is log_std_param and name not in trainable_names:
                        trainable_names.append(name)
                        break
        return trainable_names

    def trainable_params(self, net=None):
        # return the list of trainable parameters in the actor network
        net = self.actor if net is None else net
        return [p for p in net.parameters() if p.requires_grad]

    def parameters_to_vector(self, net=None):
        """
        return the vector representation of the trainable parameters in the actor network, in torch.tensor format
        :param net: which actor network to use
        :return: torch.tensor of shape (number of trainable parameters,)
        """
        net = self.actor if net is None else net
        ps = self.trainable_params(net)
        return nn.utils.parameters_to_vector(ps)

    def vector_to_parameters(self, vector, net=None):
        """
        set the trainable parameters in the actor network from the torch.tensor vector representation
        :param vector: torch.tensor of shape (number of trainable parameters,)
        :param net: which network to use
        :return:
        """
        net = self.actor if net is None else net
        ps = self.trainable_params(net)

        # safety check
        expected = sum(p.numel() for p in ps)
        if vector.numel() != expected:
            raise ValueError(f"Vector has {vector.numel()} elems, but trainable params need {expected}.")

        nn.utils.vector_to_parameters(vector, ps)

class SZPO(Base):
    def __init__(self,state_dim: int = 17, action_dim: int = 6, hidden_dim: int = 64, device=None,
                 mu = 0.03, lr = 0.1, epochs = 10, target_kl = 0.03):
        Base.__init__(self, state_dim=state_dim, action_dim=action_dim, hidden_dim=hidden_dim, device=device)

        self.lr = lr                    # learning rate
        self.mu = mu                    # parameter perturbation magnitude
        self.epochs = epochs
        self.target_kl = target_kl

        self.actor_p = ContinuousMLP(state_dim, action_dim, hidden_dim).to(self.device)             # positive perturbed actor
        self.actor_p.load_state_dict(self.actor.state_dict())
        self.actor_n = ContinuousMLP(state_dim, action_dim, hidden_dim).to(self.device)             # negative perturbed actor
        self.actor_n.load_state_dict(self.actor.state_dict())
        self.actor_optim = torch.optim.Adam(self.actor.parameters(), lr=self.lr)                    # gradient optimizer
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.actor_optim, mode='min', factor=0.3, patience=20)
        self.vec = None

    def freeze_feature_layers_for_all(self, freeze_std=False):
        model.freeze_feature_layers(actor=self.actor, freeze_std=freeze_std)
        model.freeze_feature_layers(actor=self.actor_p, freeze_std=freeze_std)
        model.freeze_feature_layers(actor=self.actor_n, freeze_std=freeze_std)

    def perturb(self):
        """
        perturb the current actor to obtain a perturbed actor, and then, perturb to the exact opposite direction to obtain the negative perturbed actor.
        :return: vec: torch.tensor of shape (number of parameters,), the parameter vector from the current actor to the perturbed actor
        """
        with torch.no_grad():
            params = self.parameters_to_vector(net=self.actor)          # parameters of the current actor

        # Sample zero-mean Gaussian noise with std = self.mu
        noise = torch.randn_like(params) * self.mu
        self.vec = noise                          # store the perturbation direction

        params_p = params + noise                # positive perturbation: actor + noise
        params_n = params - noise                # negative perturbation: actor - noise (opposite direction)

        with torch.no_grad():
            self.vector_to_parameters(params_p, net=self.actor_p)
            self.vector_to_parameters(params_n, net=self.actor_n)
        return self.vec

    def train(self, preferences):
        """
        update the gradient of the actor network using the perturbed actor and the current actor, and the human preference
        :param preferences: a list of 0,1 representing the human preference of the positively perturbed actor to the negatively current actor
        :return:
        """
        preferences = np.array(preferences)
        params = self.parameters_to_vector(net=self.actor)

        # Majority vote over preferences: +1 if positive perturbation preferred, -1 otherwise
        majority_vote = float(np.sign(np.mean(preferences)))

        # Zeroth-order gradient estimate: perturbation direction * majority vote
        gradient_estimate = self.vec * majority_vote

        # Loss whose gradient equals -gradient_estimate, enabling gradient ascent via minimization
        loss = -torch.dot(gradient_estimate.detach(), params)

        self.actor_optim.zero_grad()
        loss.backward()
        self.actor_optim.step()

        self.actor_p.load_state_dict(self.actor.state_dict())
        self.actor_n.load_state_dict(self.actor.state_dict())
        return


In [ ]:
# Load a pretrained model
model = SZPO(device='cpu')
model.load('./pretrain.pth')
model.freeze_feature_layers_for_all(freeze_std=True)
reward_ep = evaluate(model, env, deterministic=True, SEED=SEED)
print('Pretrain policy average episode reward:', round(float(np.mean(reward_ep)), 1))

In [ ]:
# Visualize the performance of pretrained model
model_pretrain = Base(device='cpu')
model_pretrain.load('./pretrain.pth')
env_show = gym.make('HalfCheetah-v5', render_mode='rgb_array', max_episode_steps=EP_LEN)
reward_ep = visualize(model_pretrain, env_show)
env_show.close()
print('Reward:', reward_ep)

The following code block starts finetuning the SZPO model, you will receive full credit if your cumulative reward evaluation is above 350. By default, if you correctly write the train() function, you should not need to finetune any hyper-parameters.

In [ ]:
# Finetune the SZPO model
reward_queue = deque(maxlen=10)
reward_curve = []
for iter in range(1000):
    model.perturb()
    preferences, returns, states, actions, log_probs, returns_p, returns_n = sample_preference(model, env)
    loss= model.train(preferences)
    reward_ep = evaluate(model, env, deterministic=True, SEED=SEED)
    reward_queue.append(float(np.mean(reward_ep)))
    model.scheduler.step(float(np.mean(reward_queue)))
    print(
            'Iteration:', iter,
            ',Episode reward:', round(float(np.mean(returns)), 1),
            ',Plus reward:', round(float(np.mean(returns_p)), 1),
            ',Minus reward:', round(float(np.mean(returns_n)), 1),
            ',Evaluation reward:', round(float(np.mean(reward_ep)), 1),
            ',Cumulative reward:', round(np.mean(reward_queue), 1),
        )
    reward_curve.append(float(np.mean(reward_queue)))
    if iter % 10 == 0:
        model.save('./szpo.pth')
        params = model.parameters_to_vector(model.actor).detach().cpu().tolist()
        with open('./szpo_weight.json', 'w') as f:
            json.dump(params, f, indent=4)
    if np.mean(reward_queue) > 350:
        model.save('./szpo.pth')
        params = model.parameters_to_vector(model.actor).detach().cpu().tolist()
        with open('./szpo_weight.json', 'w') as f:
            json.dump(params, f, indent=4)
        break

In [ ]:
# Visualize the final trained policy and plot the reward curve
env_show = gym.make('HalfCheetah-v5', render_mode='rgb_array', max_episode_steps=EP_LEN)
reward_ep = visualize(model, env_show)
env_show.close()
print('Reward:', reward_ep)

In [ ]:
# Plot the return curve
fig, ax = plt.subplots(figsize=(5, 4), dpi=200)
ax.plot(reward_curve, label='Reward')
plt.xlabel('Policy Iteration')
plt.ylabel('Cumulative Reward')
plt.legend()
plt.title('SZPO')
plt.show()